<p align="right"><a href="./C3_W2_Collaborative_RecSys_Assignment.ipynb">English</a> | <strong>中文</strong></p>

# <img align="left" src="./images/movie_camera.png"     style=" width:40px;  " > Practice lab: 协同过滤推荐系统（Collaborative Filtering Recommender Systems）

在本练习中，你将实现协同过滤（collaborative filtering），构建一个电影推荐系统。

# <img align="left" src="./images/film_reel.png"     style=" width:40px;  " > 大纲
- [ 1 - 记号 Notation](#1)
- [ 2 - 推荐系统](#2)
- [ 3 - 电影评分数据集](#3)
- [ 4 - 协同过滤学习算法](#4)
  - [ 4.1 协同过滤代价函数](#4.1)
    - [ 练习 Exercise 1](#ex01)
- [ 5 - 学习电影推荐](#5)
- [ 6 - 推荐结果](#6)
- [ 7 - 恭喜！](#7)

_**注意：** 为避免自动评分器（autograder）出错，你不能编辑或删除本实验中的非评分 cell，也请不要新增任何 cell。
**当你通过本作业后**，如果想对其中的非评分代码做实验，可以按 notebook 底部的说明操作。_

##  库 Packages <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;   " >
我们会用到现在已经熟悉的 NumPy 和 Tensorflow 库。

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 记号 Notation

|通用<br />记号  | 描述 | Python（如果有）|
|:-------------|:------------------------------------------------------------||
| $r(i,j)$     | 标量；若用户 j 给电影 i 评过分则 = 1，否则 = 0             ||
| $y(i,j)$     | 标量；用户 j 给电影 i 的评分（当 r(i,j) = 1 时才有定义） ||
|$\mathbf{w}^{(j)}$ | 向量；用户 j 的参数 ||
|$b^{(j)}$     |  标量；用户 j 的参数 ||
| $\mathbf{x}^{(i)}$ |   向量；电影 i 的特征评分        ||
| $n_u$        | 用户数 |num_users|
| $n_m$        | 电影数 | num_movies |
| $n$          | 特征数 | num_features                    |
| $\mathbf{X}$ |  由向量 $\mathbf{x}^{(i)}$ 组成的矩阵         | X |
| $\mathbf{W}$ |  由向量 $\mathbf{w}^{(j)}$ 组成的矩阵         | W |
| $\mathbf{b}$ |  由偏置参数 $b^{(j)}$ 组成的向量 | b |
| $\mathbf{R}$ | 由元素 $r(i,j)$ 组成的矩阵                    | R |

<a name="2"></a>
## 2 - 推荐系统 <img align="left" src="./images/film_rating.png" style=" width:40px;  " >
在本实验中，你将实现协同过滤学习算法，并把它应用到一个电影评分数据集上。
协同过滤推荐系统的目标是生成两类向量：对每个用户，一个体现该用户电影口味的 "参数向量（parameter vector）"；对每部电影，一个同样大小的特征向量，体现对这部电影的某种描述。这两个向量的点积再加上偏置项，应能给出该用户可能给这部电影打出的评分的估计。

下面的图详细展示了这些向量是如何学习出来的。

<figure>
   <img src="./images/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >
</figure>

已有的评分以矩阵形式提供，如图所示。$Y$ 存放评分；范围是 0.5 到 5（含），步长 0.5；若电影未被评分则为 0。$R$ 在有评分的地方取 1。电影按行排列，用户按列排列。每个用户有一个参数向量 $w^{user}$ 和一个偏置。每部电影有一个特征向量 $x^{movie}$。这些向量是同时学出来的——用已有的用户/电影评分作为训练数据。上图展示了一个训练样本：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量 $x^{movie}$ 必须同时满足所有用户，而用户向量 $w^{user}$ 必须同时满足所有电影。这正是这个方法名字的由来——所有用户协同（collaborate）生成整个评分集合。

<figure>
   <img src="./images/ColabFilterUse.PNG"  style="width:640px;height:250px;" >
</figure>

一旦学出了特征向量和参数，就可以用它们来预测某个用户会给一部未评分的电影打多少分。如上图所示。图中方程是一个例子：预测用户一对电影零的评分。

在本练习中，你将实现函数 `cofiCostFunc`，它计算协同过滤的目标函数（objective function）。实现该目标函数后，你会用一个 TensorFlow 自定义训练循环来学习协同过滤的参数。第一步是详细说明本实验将用到的数据集和数据结构。

<a name="3"></a>
## 3 - 电影评分数据集 <img align="left" src="./images/film_rating.png"     style=" width:40px;  " >
本数据集派生自 [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) 数据集。
[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集有 600 个用户对 9000 部电影的评分。为聚焦于 2000 年以后的电影，数据集规模被缩减。本数据集的评分范围是 0.5 到 5，步长 0.5。缩减后的数据集有 $n_u = 443$ 个用户、$n_m= 4778$ 部电影。

下面，你会把电影数据集加载到变量 $Y$ 和 $R$ 中。

矩阵 $Y$（一个 $n_m \times n_u$ 矩阵）存放评分 $y^{(i,j)}$。矩阵 $R$ 是一个二值指示矩阵，若用户 $j$ 给电影 $i$ 评过分则 $R(i,j) = 1$，否则 $R(i,j)=0$。

在这部分练习中，你还会用到矩阵 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$：

$$\mathbf{X} =
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} =
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} =
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$

$\mathbf{X}$ 的第 $i$ 行对应第 $i$ 部电影的特征向量 $x^{(i)}$，$\mathbf{W}$ 的第 $j$ 行对应第 $j$ 个用户的参数向量 $\mathbf{w}^{(j)}$。$x^{(i)}$ 和 $\mathbf{w}^{(j)}$ 都是 $n$ 维向量。本练习中你会取 $n=10$，因此 $\mathbf{x}^{(i)}$ 和 $\mathbf{w}^{(j)}$ 各有 10 个元素。相应地，$\mathbf{X}$ 是一个 $n_m \times 10$ 矩阵，$\mathbf{W}$ 是一个 $n_u \times 10$ 矩阵。

我们先加载电影评分数据集来理解数据的结构。
我们会用电影数据集加载 $Y$ 和 $R$。
我们还会用预先算好的值加载 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。这些值稍后会在实验里学出来，但现在我们用预先算好的值来开发代价模型（cost model）。

In [2]:
#Load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [3]:
#  From the matrix, we can compute statistics like average rating.
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 3.400 / 5


<a name="4"></a>
## 4 - 协同过滤学习算法 <img align="left" src="./images/film_filter.png"     style=" width:40px;  " >

现在，你将开始实现协同过滤学习算法。你会从实现目标函数开始。

在电影推荐这个设定下，协同过滤算法考虑一组 $n$ 维参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，模型把用户 $j$ 对电影 $i$ 的评分预测为 $y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(j)}$。给定一个由若干用户对若干电影产生的评分组成的数据集，你希望学出参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，使它们拟合得最好（即最小化平方误差）。

你将补全 cofiCostFunc 中的代码，计算协同过滤的代价函数。

<a name="4.1"></a>
### 4.1 协同过滤代价函数

协同过滤的代价函数由下式给出
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \left[ \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+ \underbrace{\left[
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
\right]}_{regularization}
\tag{1}$$
式 (1) 里的第一个求和是 "对所有满足 $r(i,j)$ 等于 $1$ 的 $i$、$j$"，也可以写成：

$$
= \left[ \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+\text{regularization}
$$

现在你应写出 cofiCostFunc（协同过滤代价函数）来返回这个代价。

<a name="ex01"></a>
### 练习 Exercise 1

**用 for 循环实现：**
先用 for 循环实现代价函数。
建议分两步来开发这个代价函数。第一步，先实现不带正则化的代价函数。下面提供了一个不含正则化的测试用例来检验你的实现。这一步跑通后，再加上正则化，并运行包含正则化的测试。注意，你应仅在 $R(i,j) = 1$ 时才累加用户 $j$、电影 $i$ 的代价。

In [5]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))            
    J = J/2
    ### END CODE HERE ### 

    return J

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    你可以像式 (1) 的求和那样，把代码组织成两层 for 循环。
    先实现不带正则化的版本。
    注意 (1) 里有些元素是向量，用 np.dot()；你也可以用 np.square()。
    仔细注意哪些元素以 i 为下标、哪些以 j 为下标。别忘了除以二。

```python
    ### START CODE HERE ###
    for j in range(nu):


        for i in range(nm):


    ### END CODE HERE ###
```
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>

    这里有更多细节。下面的代码在使用矩阵里每个元素前先把它取出来。
    也可以直接引用矩阵。
    这段代码不含正则化。

```python
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x =
            y =
            r =
            J +=
    J = J/2
    ### END CODE HERE ###

```

<details>
    <summary><font size="2" color="darkblue"><b>最后的救命稻草（完整的非正则化实现）</b></font></summary>

```python
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ###
```

<details>
    <summary><font size="2" color="darkblue"><b>正则化</b></font></summary>
     正则化就是把 W 数组和 X 数组的每个元素平方，然后把所有平方后的元素求和。
     你可以利用 np.square() 和 np.sum()。

<details>
    <summary><font size="2" color="darkblue"><b>正则化细节</b></font></summary>

```python
    J += (lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
```

</details>
</details>
</details>
</details>

In [6]:
# Reduce the data set size so that this runs faster
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


**期望输出（lambda = 0）**：
$13.67$.

In [7]:
# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


**期望输出**：

28.09

In [8]:
# Public tests
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

All tests passed!


**向量化实现**

写一个向量化实现来计算 $J$ 很重要，因为它稍后在优化过程中会被调用许多次。这里用到的线性代数不是本系列的重点，所以实现代码已给出。如果你是线性代数高手，也可以不参考下面的代码、自己写一个版本。

运行下面的代码，验证它给出的结果与非向量化版本相同。

In [9]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [10]:
# Evaluate cost function
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost: 13.67
Cost (with regularization): 28.09


**期望输出**：
Cost: 13.67
Cost (with regularization): 28.09

<a name="5"></a>
## 5 - 学习电影推荐 <img align="left" src="./images/film_man_action.png" style=" width:40px;  " >
------------------------------

在你完成协同过滤代价函数的实现后，就可以开始训练你的算法，为你自己做电影推荐了。

在下面的 cell 里，你可以输入你自己的电影选择。算法随后就会为你做推荐！我们已按我们的喜好填了一些值，但当你用我们的选择把流程跑通后，你应该把它改成符合你自己的口味。
数据集里所有电影的列表在文件 [movie list](data/small_movie_list.csv) 中。

In [12]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  Initialize my ratings

# Check the file small_movie_list.csv for id of each movie in our dataset
# For example, Toy Story 3 (2010) has ID 2700, so to rate it "5", you can set
my_ratings[2700] = 5 

#Or suppose you did not enjoy Persuasion (2007), you can set
my_ratings[2609] = 2;

# We have selected a few movies we liked / did not like and the ratings we
# gave are as follows:
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


现在，我们把这些评分加到 $Y$ 和 $R$ 里，并对评分做归一化。

In [13]:
# Reload ratings
Y, R = load_ratings_small()

# Add new user ratings to Y 
Y = np.c_[my_ratings, Y]

# Add new user indicator matrix to R
R = np.c_[(my_ratings != 0).astype(int), R]

# Normalize the Dataset
Ynorm, Ymean = normalizeRatings(Y, R)

我们来准备训练模型：初始化参数并选用 Adam 优化器。

In [14]:
#  Useful Values
num_movies, num_users = Y.shape
num_features = 100

# Set Initial Parameters (W, X), use tf.Variable to track these variables
tf.random.set_seed(1234) # for consistent results
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# Instantiate an optimizer.
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

现在我们来训练协同过滤模型。这会学出参数 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。

同时学习 $w$、$b$ 和 $x$ 所涉及的操作，并不属于 TensorFlow 神经网络包提供的那些典型 "层（layers）"。因此，Course 2 里用的 Model、Compile()、Fit()、Predict() 流程在这里并不直接适用。取而代之，我们可以用一个自定义训练循环（custom training loop）。

回忆之前实验里梯度下降的步骤。
- 重复直到收敛：
    - 计算前向传播（forward pass）
    - 计算损失相对参数的导数
    - 用学习率和算出的导数来更新参数

TensorFlow 有个了不起的能力：能替你计算这些导数。如下所示。在 `tf.GradientTape()` 段内，对 Tensorflow Variables 的操作会被跟踪。稍后调用 `tape.gradient()` 时，它会返回损失相对被跟踪变量的梯度。这些梯度随后就能用一个优化器施加到参数上。
这只是对 TensorFlow 及其他机器学习框架一个有用特性的极简介绍。想了解更多，可以在你关心的框架里查 "custom training loops"。

In [ ]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # Use TensorFlow’s GradientTape
    # to record the operations used to compute the cost 
    with tf.GradientTape() as tape:

        # Compute the cost (forward pass included in cost)
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss
    grads = tape.gradient( cost_value, [X,W,b] )

    # Run one step of gradient descent by updating
    # the value of the variables to minimize the loss.
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # Log periodically.
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 136168.7
Training loss at iteration 40: 51863.3
Training loss at iteration 60: 24598.8


<a name="6"></a>
## 6 - 推荐结果
下面，我们为所有电影和用户计算评分，并展示被推荐的电影。这些结果基于上面以 `my_ratings[]` 输入的电影和评分。要预测用户 $j$ 对电影 $i$ 的评分，你计算 $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$。所有评分都可以用矩阵乘法一次性算出。

In [ ]:
# Make a prediction using trained weights and biases
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

#restore the mean
pm = p + Ymean

my_predictions = pm[:,0]

# sort predictions
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

实践中，可以利用额外信息来增强我们的预测。上面前几百部电影的预测评分都落在一个很小的范围内。我们可以在此基础上加以增强：从那些高分电影里，挑出平均评分高、且评分数超过 20 的电影。这一节用到一个 [Pandas](https://pandas.pydata.org/) 数据框（data frame），它有许多方便的排序功能。

In [ ]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

<a name="7"></a>
## 7 - 恭喜！ <img align="left" src="./images/film_award.png"     style=" width:40px;  " >
你已经实现了一个实用的推荐系统！

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想对其中的非评分代码做实验，请点击这里。</b></font></summary>
    <p><i><b>重要提示：请务必在你已经通过本作业后再这样做，以免影响自动评分器。</b></i>
    <ol>
        <li> 在 notebook 菜单，点击 “View” > “Cell Toolbar” > “Edit Metadata”</li>
        <li> 点击你想锁定/解锁的 code cell 旁边的 “Edit Metadata” 按钮</li>
        <li> 把 “editable” 属性值设为：
            <ul>
                <li> “true” 表示解锁 </li>
                <li> “false” 表示锁定 </li>
            </ul>
        </li>
        <li> 在 notebook 菜单，点击 “View” > “Cell Toolbar” > “None” </li>
    </ol>
    <p> 下面是上述步骤的一个简短演示：
        <br>
        <img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>